In [1]:
import pickle
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd()

while not (
    (PROJECT_ROOT / "data" / "train").exists()
    and (PROJECT_ROOT / "output").exists()
):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError(
            "Could not locate amazon-ml-challenge project root."
        )
    PROJECT_ROOT = PROJECT_ROOT.parent

TRAIN_DIR = PROJECT_ROOT / "data" / "train"
OUTPUT_DIR = PROJECT_ROOT / "output"

S1_PATH = TRAIN_DIR / "train_source1.tsv"
S2_PATH = TRAIN_DIR / "train_source2.tsv"
S3_PATH = TRAIN_DIR / "train_source3.tsv"

PAIR_DIR = OUTPUT_DIR / "training_pairs_v1"

s1_idx = np.load(PAIR_DIR / "s1_idx.npy")
target_num = np.load(PAIR_DIR / "target_num.npy")
source = np.load(PAIR_DIR / "source.npy")
label = np.load(PAIR_DIR / "label.npy")
negative_route = np.load(PAIR_DIR / "negative_route.npy")

print("Training pairs:", len(label))
print("Positive:", int((label == 1).sum()))
print("Negative:", int((label == 0).sum()))

Training pairs: 5729367
Positive: 3643468
Negative: 2085899


In [3]:
def normalize_text(value):
    value = "" if value is None else str(value)

    value = unicodedata.normalize("NFKC", value)
    value = value.casefold()

    value = re.sub(
        r"[^\w\s]",
        " ",
        value,
        flags=re.UNICODE
    )

    value = re.sub(
        r"\s+",
        " ",
        value,
        flags=re.UNICODE
    )

    return value.strip()


def get_tokens(value):
    value = normalize_text(value)

    if not value:
        return []

    return value.split()


def get_digit_tokens(value):
    return set(
        re.findall(r"\d+", normalize_text(value))
    )

In [4]:
from rapidfuzz import fuzz

print(
    "RapidFuzz ready:",
    fuzz.ratio("abc ltd", "abc limited")
)

RapidFuzz ready: 77.77777777777779


In [15]:
# Cell 5 — Reconstruct the exact S1 index mapping used in training pairs
# This must follow the original train_source1 row order.

from pathlib import Path
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Locate project root independently
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd()

while not (
    (PROJECT_ROOT / "data" / "train").exists()
    and (PROJECT_ROOT / "output").exists()
):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError(
            "Could not locate amazon-ml-challenge project root."
        )
    PROJECT_ROOT = PROJECT_ROOT.parent


TRAIN_DIR = PROJECT_ROOT / "data" / "train"
OUTPUT_DIR = PROJECT_ROOT / "output"

S1_PATH = TRAIN_DIR / "train_source1.tsv"
PAIR_DIR = OUTPUT_DIR / "training_pairs_v1"


# ------------------------------------------------------------
# Load the exact 90% training S1 ID set
# ------------------------------------------------------------

TRAIN_S1_IDS_PATH = OUTPUT_DIR / "train_s1_ids.txt"

with open(
    TRAIN_S1_IDS_PATH,
    "r",
    encoding="utf-8"
) as f:
    train_s1_ids = {
        line.strip()
        for line in f
        if line.strip()
    }

print(
    "Training S1 IDs:",
    len(train_s1_ids)
)


# ------------------------------------------------------------
# Reconstruct mapping in ORIGINAL Source 1 row order
# ------------------------------------------------------------

train_s1_entity_ids = []

for chunk in pd.read_csv(
    S1_PATH,
    sep="\t",
    dtype=str,
    chunksize=100_000,
    keep_default_na=False
):

    for row in chunk.itertuples(index=False):

        if row.entity_id in train_s1_ids:
            train_s1_entity_ids.append(
                row.entity_id
            )


assert len(train_s1_entity_ids) == len(train_s1_ids)
assert len(set(train_s1_entity_ids)) == len(train_s1_entity_ids)


train_s1_id_to_idx = {
    entity_id: idx
    for idx, entity_id in enumerate(
        train_s1_entity_ids
    )
}

train_s1_idx_to_id = {
    idx: entity_id
    for entity_id, idx in train_s1_id_to_idx.items()
}


# ------------------------------------------------------------
# Load training pair arrays
# ------------------------------------------------------------

s1_idx = np.load(
    PAIR_DIR / "s1_idx.npy"
)

target_num = np.load(
    PAIR_DIR / "target_num.npy"
)

source = np.load(
    PAIR_DIR / "source.npy"
)

label = np.load(
    PAIR_DIR / "label.npy"
)

negative_route = np.load(
    PAIR_DIR / "negative_route.npy"
)


# ------------------------------------------------------------
# Verify index compatibility
# ------------------------------------------------------------

assert s1_idx.min() >= 0
assert s1_idx.max() < len(train_s1_entity_ids)

print("\nS1 mapping:", len(train_s1_id_to_idx))
print("Training pair rows:", len(label))
print("Positive:", int((label == 1).sum()))
print("Negative:", int((label == 0).sum()))

print(
    "\nS1 index mapping verification: PASS"
)

Training S1 IDs: 1986139

S1 mapping: 1986139
Training pair rows: 5729367
Positive: 3643468
Negative: 2085899

S1 index mapping verification: PASS


In [16]:
from rapidfuzz import fuzz
import re

In [17]:
def text_tokens(value):
    if not value:
        return set()
    return set(value.split())


def digit_tokens(value):
    if not value:
        return set()
    return set(re.findall(r"\d+", value))


def safe_length_ratio(a, b):
    if not a or not b:
        return 0.0

    return (
        min(len(a), len(b))
        / max(len(a), len(b))
    )


def token_jaccard(a_tokens, b_tokens):
    if not a_tokens and not b_tokens:
        return 0.0

    union = a_tokens | b_tokens

    if not union:
        return 0.0

    return len(a_tokens & b_tokens) / len(union)

In [7]:
feature_order_s2 = np.flatnonzero(source == 2)
feature_order_s3 = np.flatnonzero(source == 3)

feature_order_s2 = feature_order_s2[
    np.argsort(
        target_num[feature_order_s2],
        kind="stable"
    )
]

feature_order_s3 = feature_order_s3[
    np.argsort(
        target_num[feature_order_s3],
        kind="stable"
    )
]

print("S2 pair rows:", len(feature_order_s2))
print("S3 pair rows:", len(feature_order_s3))

S2 pair rows: 2819082
S3 pair rows: 2910285


In [18]:
FEATURE_NAMES = [
    "name_exact",
    "name_ratio",
    "name_token_ratio",
    "name_token_jaccard",
    "name_length_ratio",
    "name_digit_overlap",

    "address_exact",
    "address_ratio",
    "address_token_ratio",
    "address_token_jaccard",
    "address_length_ratio",
    "address_digit_overlap",

    "country_same",
    "name_present_both",
    "address_present_both",
]

In [19]:
def calculate_pair_features(
    s1_name,
    s1_address,
    s1_country,
    target_name,
    target_address,
    target_country
):
    # -----------------------------
    # Name
    # -----------------------------

    name_exact = int(
        bool(s1_name)
        and s1_name == target_name
    )

    name_ratio = (
        fuzz.ratio(
            s1_name,
            target_name
        ) / 100.0
        if s1_name and target_name
        else 0.0
    )

    name_token_ratio = (
        fuzz.token_set_ratio(
            s1_name,
            target_name
        ) / 100.0
        if s1_name and target_name
        else 0.0
    )

    s1_name_tokens = text_tokens(s1_name)
    target_name_tokens = text_tokens(target_name)

    name_token_jaccard = token_jaccard(
        s1_name_tokens,
        target_name_tokens
    )

    name_length_ratio = safe_length_ratio(
        s1_name,
        target_name
    )

    name_digit_overlap = len(
        digit_tokens(s1_name)
        &
        digit_tokens(target_name)
    )

    # -----------------------------
    # Address
    # -----------------------------

    address_exact = int(
        bool(s1_address)
        and s1_address == target_address
    )

    address_ratio = (
        fuzz.ratio(
            s1_address,
            target_address
        ) / 100.0
        if s1_address and target_address
        else 0.0
    )

    address_token_ratio = (
        fuzz.token_set_ratio(
            s1_address,
            target_address
        ) / 100.0
        if s1_address and target_address
        else 0.0
    )

    s1_address_tokens = text_tokens(s1_address)
    target_address_tokens = text_tokens(target_address)

    address_token_jaccard = token_jaccard(
        s1_address_tokens,
        target_address_tokens
    )

    address_length_ratio = safe_length_ratio(
        s1_address,
        target_address
    )

    address_digit_overlap = len(
        digit_tokens(s1_address)
        &
        digit_tokens(target_address)
    )

    # -----------------------------
    # Other
    # -----------------------------

    country_same = int(
        bool(s1_country)
        and s1_country == target_country
    )

    name_present_both = int(
        bool(s1_name and target_name)
    )

    address_present_both = int(
        bool(s1_address and target_address)
    )

    return np.array(
        [
            name_exact,
            name_ratio,
            name_token_ratio,
            name_token_jaccard,
            name_length_ratio,
            name_digit_overlap,

            address_exact,
            address_ratio,
            address_token_ratio,
            address_token_jaccard,
            address_length_ratio,
            address_digit_overlap,

            country_same,
            name_present_both,
            address_present_both,
        ],
        dtype=np.float32
    )

In [28]:
N_PAIRS = len(label)
N_FEATURES = len(FEATURE_NAMES)

features = np.empty(
    (N_PAIRS, N_FEATURES),
    dtype=np.float32
)

print("Pairs:", N_PAIRS)
print("Features:", N_FEATURES)
print("Memory:", f"{features.nbytes / 1024**2:.1f} MB")

Pairs: 5729367
Features: 15
Memory: 327.8 MB


In [29]:
def generate_features_for_source(
    source_path,
    pair_indices,
    source_name
):
    pair_indices = np.asarray(
        pair_indices,
        dtype=np.int64
    )

    # ------------------------------------------------------------
    # The pair indices are sorted by target_num.
    # The source TSV itself is NOT assumed to be sorted.
    # ------------------------------------------------------------

    sorted_target_numbers = target_num[
        pair_indices
    ]

    filled = 0
    total_rows = 0

    for chunk_number, chunk in enumerate(
        pd.read_csv(
            source_path,
            sep="\t",
            dtype=str,
            chunksize=100_000,
            keep_default_na=False
        ),
        start=1
    ):

        total_rows += len(chunk)

        # --------------------------------------------------------
        # Convert target entity IDs to their numeric suffixes
        # --------------------------------------------------------

        chunk_target_numbers = (
            chunk["entity_id"]
            .str.slice(3)
            .astype(np.int64)
            .to_numpy()
        )

        # --------------------------------------------------------
        # Find the pair-index range for every target in this chunk
        #
        # Because sorted_target_numbers is sorted, searchsorted
        # gives the exact [left, right) range of pairs for a target.
        # --------------------------------------------------------

        left_positions = np.searchsorted(
            sorted_target_numbers,
            chunk_target_numbers,
            side="left"
        )

        right_positions = np.searchsorted(
            sorted_target_numbers,
            chunk_target_numbers,
            side="right"
        )

        # --------------------------------------------------------
        # Process only target rows that actually have pairs
        # --------------------------------------------------------

        for row_position, row in enumerate(
            chunk.itertuples(index=False)
        ):

            left = int(
                left_positions[row_position]
            )

            right = int(
                right_positions[row_position]
            )

            if left == right:
                continue

            target_name = normalize_text(
                row.business_name
            )

            target_address = normalize_text(
                row.business_address
            )

            target_country = (
                row.country.casefold().strip()
            )

            # ----------------------------------------------------
            # All training pairs associated with this target
            # ----------------------------------------------------

            for sorted_position in range(left, right):

                pair_idx = pair_indices[
                    sorted_position
                ]

                s1_index = s1_idx[
                    pair_idx
                ]

                features[pair_idx] = (
                    calculate_pair_features(
                        s1_name[s1_index],
                        s1_address[s1_index],
                        s1_country[s1_index],

                        target_name,
                        target_address,
                        target_country
                    )
                )

                filled += 1

        if chunk_number % 10 == 0:
            print(
                f"{source_name}: "
                f"{total_rows:,} target rows | "
                f"{filled:,} pair features"
            )

    print(
        f"{source_name} COMPLETE | "
        f"target rows={total_rows:,} | "
        f"pairs filled={filled:,}"
    )

    assert filled == len(pair_indices), (
        f"{source_name}: expected "
        f"{len(pair_indices):,} features, "
        f"but filled {filled:,}"
    )

    return filled

In [30]:
s2_indices = np.flatnonzero(source == 2)
s3_indices = np.flatnonzero(source == 3)

s2_indices = s2_indices[
    np.argsort(
        target_num[s2_indices],
        kind="stable"
    )
]

s3_indices = s3_indices[
    np.argsort(
        target_num[s3_indices],
        kind="stable"
    )
]

print("S2 training pairs:", len(s2_indices))
print("S3 training pairs:", len(s3_indices))

S2 training pairs: 2819082
S3 training pairs: 2910285


In [31]:
filled_s2 = generate_features_for_source(
    S2_PATH,
    s2_indices,
    "S2"
)

filled_s3 = generate_features_for_source(
    S3_PATH,
    s3_indices,
    "S3"
)

print(
    "Total filled:",
    filled_s2 + filled_s3
)

print(
    "Expected:",
    N_PAIRS
)

S2: 1,000,000 target rows | 724,534 pair features
S2: 2,000,000 target rows | 1,324,530 pair features
S2: 3,000,000 target rows | 1,856,039 pair features
S2: 4,000,000 target rows | 2,343,619 pair features
S2: 5,000,000 target rows | 2,803,491 pair features
S2 COMPLETE | target rows=5,034,616 | pairs filled=2,819,082
S3: 1,000,000 target rows | 719,564 pair features
S3: 2,000,000 target rows | 1,317,578 pair features
S3: 3,000,000 target rows | 1,845,494 pair features
S3: 4,000,000 target rows | 2,328,716 pair features
S3: 5,000,000 target rows | 2,783,856 pair features
S3 COMPLETE | target rows=5,285,603 | pairs filled=2,910,285
Total filled: 5729367
Expected: 5729367


In [32]:
assert features.shape == (
    N_PAIRS,
    N_FEATURES
)

assert np.isfinite(features).all()

print("Feature matrix:", features.shape)
print("NaNs:", np.isnan(features).sum())
print("Infs:", np.isinf(features).sum())

for i, name in enumerate(FEATURE_NAMES):
    print(
        f"{name:25s}"
        f" min={features[:, i].min():.4f}"
        f" max={features[:, i].max():.4f}"
        f" mean={features[:, i].mean():.4f}"
    )

print("\nFeature validation: PASS")

Feature matrix: (5729367, 15)
NaNs: 0
Infs: 0
name_exact                min=0.0000 max=1.0000 mean=0.0000
name_ratio                min=0.0000 max=1.0000 mean=0.3191
name_token_ratio          min=0.0000 max=1.0000 mean=0.3226
name_token_jaccard        min=0.0000 max=1.0000 mean=0.0159
name_length_ratio         min=0.0233 max=1.0000 mean=0.6850
name_digit_overlap        min=0.0000 max=1.0000 mean=0.0000
address_exact             min=0.0000 max=1.0000 mean=0.0000
address_ratio             min=0.0000 max=1.0000 mean=0.3119
address_token_ratio       min=0.0000 max=1.0000 mean=0.3312
address_token_jaccard     min=0.0000 max=1.0000 mean=0.0088
address_length_ratio      min=0.0000 max=1.0000 mean=0.6289
address_digit_overlap     min=0.0000 max=6.0000 mean=0.0179
country_same              min=0.0000 max=1.0000 mean=0.5182
name_present_both         min=1.0000 max=1.0000 mean=1.0000
address_present_both      min=0.0000 max=1.0000 mean=0.9629

Feature validation: PASS


In [33]:
feature_summary = []

for i, name in enumerate(FEATURE_NAMES):

    positive_mean = features[
        label == 1,
        i
    ].mean()

    negative_mean = features[
        label == 0,
        i
    ].mean()

    feature_summary.append({
        "feature": name,
        "positive_mean": positive_mean,
        "negative_mean": negative_mean,
        "difference": (
            positive_mean - negative_mean
        )
    })

feature_summary_df = pd.DataFrame(
    feature_summary
)

feature_summary_df.sort_values(
    "difference",
    ascending=False
)

,feature,positive_mean,negative_mean,difference
12,country_same,5.202793e-01,0.514562,5.716980e-03
4,name_length_ratio,6.852216e-01,0.684504,7.178187e-04
5,name_digit_overlap,1.042962e-05,0.000006,4.676708e-06
6,address_exact,2.744638e-07,0.000000,2.744638e-07
13,name_present_both,1.000000e+00,1.000000,0.000000e+00
0,name_exact,1.372319e-06,0.000002,-1.024729e-06
9,address_token_jaccard,8.726970e-03,0.008908,-1.813499e-04
11,address_digit_overlap,1.773695e-02,0.018196,-4.585646e-04
3,name_token_jaccard,1.384263e-02,0.019402,-5.559118e-03
7,address_ratio,3.095760e-01,0.315993,-6.417125e-03


In [34]:
FEATURE_DIR = OUTPUT_DIR / "training_features_v1"

FEATURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

np.save(
    FEATURE_DIR / "features.npy",
    features
)

np.save(
    FEATURE_DIR / "labels.npy",
    label
)

np.save(
    FEATURE_DIR / "s1_idx.npy",
    s1_idx
)

np.save(
    FEATURE_DIR / "target_num.npy",
    target_num
)

np.save(
    FEATURE_DIR / "source.npy",
    source
)

with open(
    FEATURE_DIR / "feature_names.pkl",
    "wb"
) as f:
    pickle.dump(
        FEATURE_NAMES,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

print(
    "Saved feature artifacts:",
    FEATURE_DIR
)

Saved feature artifacts: /Users/bhavikesh/Documents/Hackathon/Amazon_ml/amazon-ml-challenge/output/training_features_v1
